# 03 公共谱图二次审计与训练谱图清单冻结

本Notebook只读取冻结拆分和现有谱图文件，构建谱图级可追溯清单。主分析使用20 ppm；50/100 ppm只作为敏感性分析。ChemVista谱图始终不进入训练。

**Legacy限制：** `EnvMolBench_Cleaned.pkl`没有保存公共库原始谱图ID和仪器字段。本Notebook使用`来源库 + 接受序号 + 文件偏移`建立稳定谱图ID，并将无法恢复的字段明确标记为WARN，不进行猜测。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pickle
import platform
import sys

import numpy as np
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'pbt_msms_v1').exists():
    candidates = [ROOT, *ROOT.parents]
    ROOT = next((p for p in candidates if (p / 'outputs' / 'pbt_msms_v1').exists()), ROOT)

SPLIT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'split_v1'
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'spectrum_audit_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'split_manifest_csv': SPLIT_DIR / 'compound_split_manifest.csv',
    'split_freeze_json': SPLIT_DIR / 'split_freeze_manifest.json',
    'split_qc_csv': SPLIT_DIR / 'leakage_checks.csv',
    'public_stream': ROOT / '质谱数据' / 'EnvMolBench_Cleaned.pkl',
    'public_index': ROOT / '质谱数据' / 'EnvMolBench_Cleaned_Index.pkl',
    'public_cleaning_notebook': ROOT / '质谱数据' / '质谱数据清洗.ipynb',
    'public_cleaning_module': ROOT / 'src' / 'pbt_msms' / 'public_spectrum_cleaning.py',
    'public_summary': ROOT / '质谱数据' / 'EnvMolBench_Cleaned_Summary.json',
    'chemvista_index': ROOT / '验证实验谱图数据' / 'ChemVista_cleaned' / 'ChemVista_spectrum_index.pkl',
    'chemvista_peaks': ROOT / '验证实验谱图数据' / 'ChemVista_cleaned' / 'ChemVista_spectrum_peaks.pkl',
    'chemvista_summary': ROOT / '验证实验谱图数据' / 'ChemVista_cleaned' / 'ChemVista_processing_summary.json',
}
required_names = [name for name in PATHS if name != 'public_summary']
missing = [str(PATHS[name]) for name in required_names if not PATHS[name].exists()]
assert not missing, f'缺少输入文件: {missing}'
print('Python:', sys.executable)
print('输出目录:', OUTPUT_DIR)

In [ ]:
LEGACY_LIBRARY_COUNTS = [('GNPS', 1_493_068), ('MassBank', 85_787), ('MoNA', 67_473)]
LIBRARY_PRIORITY = {'MassBank': 0, 'MoNA': 1, 'GNPS': 2}
ADDUCTS = {
    '[M+H]+': (1.007276, 1), '[M+Na]+': (22.989769, 1), '[M+NH4]+': (18.033823, 1),
    '[M+K]+': (38.963158, 1), '[M-H2O+H]+': (-17.003284, 1), '[M]+.': (-0.0005485, 1),
    '[M-H]-': (-1.007276, 1), '[M+Cl]-': (34.968853, 1), '[M+FA]-': (44.99765, 1),
    '[M+Ac]-': (59.01330, 1), '[M-H2O-H]-': (-19.01784, 1), '[M]-.': (0.0005485, 1),
}

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def normalize_ion_mode(value) -> str:
    text = str(value).strip().lower()
    if text in {'positive', 'pos', '+', '1', '1+'} or 'positive' in text:
        return 'Positive'
    if text in {'negative', 'neg', '-', '-1', '1-'} or 'negative' in text:
        return 'Negative'
    return 'Unknown'

def standardize_peaks(mz, intensity, max_fragments: int = 119):
    mz = np.asarray(mz, dtype=np.float64)
    intensity = np.asarray(intensity, dtype=np.float64)
    valid = np.isfinite(mz) & np.isfinite(intensity) & (mz >= 15.0) & (mz <= 2000.0) & (intensity > 0)
    mz, intensity = mz[valid], intensity[valid]
    if len(mz) == 0:
        return mz, intensity
    intensity = intensity / intensity.max()
    keep = intensity >= 0.001
    mz, intensity = mz[keep], intensity[keep]
    order = np.argsort(mz, kind='mergesort')
    mz, intensity = mz[order], intensity[order]
    isotope_keep = np.ones(len(mz), dtype=bool)
    for i in range(len(mz)):
        if not isotope_keep[i]:
            continue
        for j in range(i + 1, len(mz)):
            delta = mz[j] - mz[i]
            if delta > 1.01:
                break
            if 1.00085 < delta < 1.00585 and intensity[j] < intensity[i] * 0.8:
                isotope_keep[j] = False
    mz, intensity = mz[isotope_keep], intensity[isotope_keep]
    if len(mz) > max_fragments:
        strongest = np.argsort(intensity, kind='mergesort')[::-1][:max_fragments]
        mz, intensity = mz[strongest], intensity[strongest]
    order = np.argsort(mz, kind='mergesort')
    return mz[order], intensity[order]

def peak_hash(mz, intensity) -> str:
    if len(mz) == 0:
        return ''
    values = np.column_stack([np.round(mz, 4), np.round(intensity, 5)]).astype('<f8', copy=False)
    return hashlib.sha256(values.tobytes()).hexdigest()

def spectrum_hash(precursor_mz, ion_mode, p_hash) -> str:
    payload = f'{normalize_ion_mode(ion_mode)}|{float(precursor_mz):.4f}|{p_hash}'
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def coarse_signature(precursor_mz, ion_mode, mz, intensity) -> str:
    if len(mz) == 0:
        return ''
    top = np.argsort(intensity, kind='mergesort')[::-1][:20]
    bins = ','.join(f'{x:.1f}' for x in np.sort(np.round(mz[top], 1)))
    payload = f'{normalize_ion_mode(ion_mode)}|{float(precursor_mz):.1f}|{bins}'
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def modified_cosine_score(mz_a, intensity_a, precursor_a, mz_b, intensity_b, precursor_b, tolerance=0.02):
    mz_a = np.asarray(mz_a, dtype=float)
    mz_b = np.asarray(mz_b, dtype=float)
    intensity_a = np.sqrt(np.clip(np.asarray(intensity_a, dtype=float), 0, None))
    intensity_b = np.sqrt(np.clip(np.asarray(intensity_b, dtype=float), 0, None))
    norm_a, norm_b = np.linalg.norm(intensity_a), np.linalg.norm(intensity_b)
    if not len(mz_a) or not len(mz_b) or norm_a == 0 or norm_b == 0:
        return 0.0, 0
    intensity_a, intensity_b = intensity_a / norm_a, intensity_b / norm_b
    precursor_shift = float(precursor_b) - float(precursor_a)
    candidates = []
    for i, mass_a in enumerate(mz_a):
        for shift in (0.0, precursor_shift):
            target = mass_a + shift
            left = int(np.searchsorted(mz_b, target - tolerance, side='left'))
            right = int(np.searchsorted(mz_b, target + tolerance, side='right'))
            for j in range(left, right):
                candidates.append((float(intensity_a[i] * intensity_b[j]), i, j))
    used_a, used_b, score, matches = set(), set(), 0.0, 0
    for product, i, j in sorted(candidates, reverse=True):
        if i in used_a or j in used_b:
            continue
        used_a.add(i); used_b.add(j); score += product; matches += 1
    return float(min(score, 1.0)), int(matches)

mass_cache = {}
def calculate_ppm(smiles, precursor_mz, adduct):
    if adduct not in ADDUCTS or not np.isfinite(float(precursor_mz)):
        return np.nan
    if smiles not in mass_cache:
        mol = Chem.MolFromSmiles(str(smiles))
        mass_cache[smiles] = Descriptors.ExactMolWt(mol) if mol is not None else np.nan
    neutral_mass = mass_cache[smiles]
    if not np.isfinite(neutral_mass):
        return np.nan
    shift, charge = ADDUCTS[adduct]
    theoretical = (neutral_mass + shift) / charge
    return abs(float(precursor_mz) - theoretical) / theoretical * 1e6

def ppm_tier(value):
    if pd.isna(value): return 'unknown'
    if value <= 20: return 'main_20ppm'
    if value <= 50: return 'sensitivity_50ppm'
    if value <= 100: return 'sensitivity_100ppm'
    return 'over_100ppm'

## 1. 读取冻结拆分并恢复公共谱图库来源边界

In [ ]:
split = pd.read_csv(PATHS['split_manifest_csv'])
split['inchikey_2d'] = split['inchikey_2d'].astype(str)
split_by_key = split.set_index('inchikey_2d')
split_qc = pd.read_csv(PATHS['split_qc_csv'])
with PATHS['split_freeze_json'].open('r', encoding='utf-8') as handle:
    split_freeze = json.load(handle)
with PATHS['public_index'].open('rb') as handle:
    public_index = pickle.load(handle)
with PATHS['public_stream'].open('rb') as handle:
    stream_header = pickle.load(handle)
header_config = stream_header[1] if isinstance(stream_header, tuple) and len(stream_header) == 2 else {}
public_schema_version = str(header_config.get('schema_version', 'LEGACY'))
is_traceable_v2 = public_schema_version == 'ENVMOLBENCH_CLEANED_V2'
if is_traceable_v2:
    assert PATHS['public_summary'].is_file(), 'V2公共谱图缺少清洗摘要，禁止继续审计'
    with PATHS['public_summary'].open('r', encoding='utf-8') as handle:
        public_cleaning_summary = json.load(handle)
    assert public_cleaning_summary['status'] == 'PASS'
    LIBRARY_COUNTS = [(library, int(counts.get('SUCCESS', 0))) for library, counts in public_cleaning_summary['counts_by_library'].items()]
    expected_count = int(public_cleaning_summary['counts']['SUCCESS'])
else:
    public_cleaning_summary = None
    LIBRARY_COUNTS = LEGACY_LIBRARY_COUNTS
    expected_count = sum(count for _, count in LIBRARY_COUNTS)

all_offset_count = sum(len(v) for v in public_index.values())
assert all_offset_count == expected_count, (all_offset_count, expected_count)
all_offsets = np.fromiter((offset for values in public_index.values() for offset in values), dtype=np.int64, count=all_offset_count)
all_offsets.sort()

library_starts = {}
cursor = 0
for library, count in LIBRARY_COUNTS:
    library_starts[library] = (cursor, cursor + count)
    cursor += count

def library_from_rank(rank):
    for library, (start, stop) in library_starts.items():
        if start <= rank < stop:
            return library, rank - start + 1
    raise IndexError(rank)

public_keys = set(split.loc[split['has_public_spectrum'].astype(str).str.lower().isin({'true', '1'}), 'inchikey_2d'])
requested = sorted((offset, key) for key in public_keys for offset in public_index.get(key, []))
print('公共任务结构:', len(public_keys))
print('需要随机访问的公共谱图:', len(requested))
print('公共全库谱图:', all_offset_count)

## 2. 提取任务相关公共谱图、重算ppm并生成统一峰哈希

In [ ]:
public_rows = []
key_mismatch_count = 0
with PATHS['public_stream'].open('rb') as handle:
    stream_header = pickle.load(handle)
    for offset, expected_key in tqdm(requested, desc='读取任务相关公共谱图', mininterval=2.0):
        handle.seek(offset)
        actual_key, record = pickle.load(handle)
        if actual_key != expected_key:
            key_mismatch_count += 1
            continue
        global_rank = int(np.searchsorted(all_offsets, offset))
        inferred_library, accepted_rank = library_from_rank(global_rank)
        stored_library = str(record.get('source_library', '')).strip()
        library = stored_library if stored_library in LIBRARY_PRIORITY else inferred_library
        source_library_status = 'stored_in_traceable_v2_record' if stored_library in LIBRARY_PRIORITY else 'inferred_from_legacy_append_rank_validated_by_total_counts'
        source_record_ordinal = record.get('source_record_ordinal', accepted_rank)
        mask = ~np.asarray(record['padding_mask'], dtype=bool)
        fragment_mask = mask.copy()
        fragment_mask[0] = False
        tensor = np.asarray(record['tensor'])
        mz = tensor[fragment_mask, 0].astype(np.float64) * 2000.0
        intensity = tensor[fragment_mask, 3].astype(np.float64)
        mz, intensity = standardize_peaks(mz, intensity)
        p_hash = peak_hash(mz, intensity)
        precursor_mz = float(record['precursor_mz'])
        ion_mode = normalize_ion_mode(record.get('ionmode'))
        stored_ppm = pd.to_numeric(pd.Series([record.get('identity_ppm_error')]), errors='coerce').iloc[0]
        ppm_error = float(stored_ppm) if np.isfinite(stored_ppm) else calculate_ppm(record.get('smiles'), precursor_mz, record.get('adduct'))
        compound = split_by_key.loc[actual_key]
        public_rows.append({
            'spectrum_id': f'ENVMB:{library}:{int(source_record_ordinal):09d}' if is_traceable_v2 else f'ENVMB:{library}:{accepted_rank:07d}',
            'source_library': library,
            'source_library_status': source_library_status,
            'accepted_rank_in_library': accepted_rank,
            'stream_offset': int(offset),
            'source_record_ordinal': int(source_record_ordinal),
            'original_spectrum_id': record.get('original_spectrum_id', pd.NA),
            'original_spectrum_id_status': record.get('original_spectrum_id_status', 'unavailable_in_legacy_export'),
            'instrument': record.get('instrument') or pd.NA,
            'instrument_status': record.get('instrument_status', 'unavailable_in_legacy_export'),
            'record_schema_version': record.get('schema_version', 'LEGACY'),
            'inchikey_2d': actual_key,
            'precursor_mz': precursor_mz,
            'ion_mode': ion_mode,
            'adduct': record.get('adduct'),
            'collision_energy_value': record.get('ce_real', -1.0),
            'collision_energy_bucket': record.get('ce_bucket', -1.0),
            'cleaned_peak_n': int(len(mz)),
            'peak_hash_common': p_hash,
            'spectrum_hash_common': spectrum_hash(precursor_mz, ion_mode, p_hash),
            'near_duplicate_signature': coarse_signature(precursor_mz, ion_mode, mz, intensity),
            'identity_ppm_error': ppm_error,
            'ppm_tier': ppm_tier(ppm_error),
            'supervised_compound_role': compound['supervised_compound_role'],
            'chemvista_evaluation_role': compound['chemvista_evaluation_role'],
            'cv_group': compound['cv_group'],
            'cv_fold': compound['cv_fold'],
        })
public = pd.DataFrame(public_rows)
assert len(public) == len(requested) - key_mismatch_count
print('提取完成:', len(public), 'key mismatch:', key_mismatch_count)
display(public[['source_library', 'ppm_tier']].value_counts().rename('n_spectra').reset_index())

## 3. 公共谱图精确去重、训练资格与Strict资格

In [ ]:
public['library_priority'] = public['source_library'].map(LIBRARY_PRIORITY).astype(int)
public = public.sort_values(['inchikey_2d', 'spectrum_hash_common', 'library_priority', 'spectrum_id']).reset_index(drop=True)
same_identity_key = public['inchikey_2d'] + '|' + public['spectrum_hash_common']
public['exact_duplicate_same_identity'] = same_identity_key.duplicated(keep=False)
public['exact_dedup_keep'] = ~same_identity_key.duplicated(keep='first')
identity_n = public.groupby('spectrum_hash_common')['inchikey_2d'].transform('nunique')
public['cross_identity_exact_collision'] = identity_n.gt(1)

is_development = public['supervised_compound_role'].eq('development')
is_transfer_structure = public['chemvista_evaluation_role'].eq('transfer')
base_quality = public['exact_dedup_keep'] & ~public['cross_identity_exact_collision']
public['primary_train_eligible_20ppm'] = is_development & base_quality & public['identity_ppm_error'].le(20)
public['primary_train_eligible_50ppm'] = is_development & base_quality & public['identity_ppm_error'].le(50)
public['primary_train_eligible_100ppm'] = is_development & base_quality & public['identity_ppm_error'].le(100)
public['strict_train_eligible_20ppm'] = public['primary_train_eligible_20ppm'] & ~is_transfer_structure
public['strict_train_eligible_50ppm'] = public['primary_train_eligible_50ppm'] & ~is_transfer_structure
public['strict_train_eligible_100ppm'] = public['primary_train_eligible_100ppm'] & ~is_transfer_structure
public['excluded_from_supervised_reason'] = ''
public.loc[~is_development, 'excluded_from_supervised_reason'] = 'external_novel_compound'
public.loc[public['cross_identity_exact_collision'], 'excluded_from_supervised_reason'] = 'cross_identity_exact_spectrum_collision'
public.loc[~public['exact_dedup_keep'] & ~public['cross_identity_exact_collision'], 'excluded_from_supervised_reason'] = 'same_identity_exact_duplicate'
public.loc[is_development & base_quality & public['identity_ppm_error'].gt(20), 'excluded_from_supervised_reason'] = 'over_20ppm_main_only'

dedup_summary = pd.DataFrame([
    {'metric': 'public_relevant_spectra', 'value': len(public)},
    {'metric': 'same_identity_exact_duplicate_spectra', 'value': int(public['exact_duplicate_same_identity'].sum())},
    {'metric': 'cross_identity_exact_collision_spectra', 'value': int(public['cross_identity_exact_collision'].sum())},
    {'metric': 'primary_train_20ppm_spectra', 'value': int(public['primary_train_eligible_20ppm'].sum())},
    {'metric': 'strict_train_20ppm_spectra', 'value': int(public['strict_train_eligible_20ppm'].sum())},
])
display(dedup_summary)

## 4. ChemVista统一哈希及公共训练谱图碰撞审计

In [ ]:
chem_index = pd.read_pickle(PATHS['chemvista_index']).reset_index(drop=True)
with PATHS['chemvista_peaks'].open('rb') as handle:
    chem_peaks = pickle.load(handle)
chem_roles = split_by_key['chemvista_evaluation_role'].to_dict()
chem_index['chemvista_evaluation_role'] = chem_index['inchikey_2d'].map(chem_roles).fillna('none')
chem = chem_index.loc[chem_index['chemvista_evaluation_role'].isin(['natural_novel', 'selected_novel', 'transfer'])].copy()
chem_rows = []
for _, row in tqdm(chem.iterrows(), total=len(chem), desc='处理ChemVista统一哈希', mininterval=2.0):
    peaks = chem_peaks[row['spectrum_id']]
    mz, intensity = standardize_peaks(peaks['mz'], peaks.get('intensity_norm_100', peaks['intensity_raw']))
    p_hash = peak_hash(mz, intensity)
    role = row['chemvista_evaluation_role']
    chem_rows.append({
        'spectrum_id': row['spectrum_id'],
        'source_library': 'ChemVista',
        'source_file': row['source_file'],
        'source_relpath': row['source_relpath'],
        'file_sha256': row['file_sha256'],
        'original_accession': row['accession'],
        'inchikey_2d': row['inchikey_2d'],
        'precursor_mz': float(row['precursor_mz']),
        'ion_mode': normalize_ion_mode(row['ion_mode']),
        'adduct': row['precursor_type'],
        'collision_energy_value': row['collision_energy_value'],
        'instrument': row['instrument'],
        'instrument_type': row['instrument_type'],
        'cleaned_peak_n_common': int(len(mz)),
        'peak_hash_common': p_hash,
        'spectrum_hash_common': spectrum_hash(row['precursor_mz'], row['ion_mode'], p_hash),
        'near_duplicate_signature': coarse_signature(row['precursor_mz'], row['ion_mode'], mz, intensity),
        'chemvista_evaluation_role': role,
        'evaluation_claim': 'unseen_compound' if role in {'natural_novel', 'selected_novel'} else 'measurement_domain_transfer',
    })
chem_eval = pd.DataFrame(chem_rows)
primary_hashes = set(public.loc[public['primary_train_eligible_20ppm'], 'spectrum_hash_common'])
primary_coarse = set(public.loc[public['primary_train_eligible_20ppm'], 'near_duplicate_signature'])
chem_eval['exact_public_train_spectrum_collision'] = chem_eval['spectrum_hash_common'].isin(primary_hashes)
chem_eval['near_public_train_candidate'] = chem_eval['near_duplicate_signature'].isin(primary_coarse)
candidate_signatures = set(chem_eval.loc[chem_eval['near_public_train_candidate'], 'near_duplicate_signature'])
public_near_candidates = public.loc[
    public['primary_train_eligible_20ppm'] & public['near_duplicate_signature'].isin(candidate_signatures),
].copy()
public_peak_cache = {}
with PATHS['public_stream'].open('rb') as handle:
    for row in public_near_candidates.itertuples(index=False):
        handle.seek(int(row.stream_offset))
        _, record = pickle.load(handle)
        tensor = np.asarray(record['tensor'])
        mask = ~np.asarray(record['padding_mask'], dtype=bool)
        mask[0] = False
        mz, intensity = standardize_peaks(tensor[mask, 0].astype(float) * 2000.0, tensor[mask, 3].astype(float))
        public_peak_cache[row.spectrum_id] = (mz, intensity)
near_pair_rows = []
for chem_row in chem_eval.loc[chem_eval['near_public_train_candidate']].itertuples(index=False):
    chem_payload = chem_peaks[chem_row.spectrum_id]
    chem_mz, chem_intensity = standardize_peaks(chem_payload['mz'], chem_payload.get('intensity_norm_100', chem_payload['intensity_raw']))
    candidates = public_near_candidates.loc[public_near_candidates['near_duplicate_signature'].eq(chem_row.near_duplicate_signature)]
    for public_row in candidates.itertuples(index=False):
        public_mz, public_intensity = public_peak_cache[public_row.spectrum_id]
        score, matched_peaks = modified_cosine_score(
            chem_mz, chem_intensity, chem_row.precursor_mz,
            public_mz, public_intensity, public_row.precursor_mz,
        )
        near_pair_rows.append({
            'chemvista_spectrum_id': chem_row.spectrum_id, 'chemvista_ik2d': chem_row.inchikey_2d,
            'chemvista_evaluation_role': chem_row.chemvista_evaluation_role,
            'public_spectrum_id': public_row.spectrum_id, 'public_ik2d': public_row.inchikey_2d,
            'same_ik2d': chem_row.inchikey_2d == public_row.inchikey_2d,
            'modified_cosine': score, 'matched_peaks': matched_peaks,
        })
modified_cosine_pair_audit = pd.DataFrame(near_pair_rows, columns=[
    'chemvista_spectrum_id', 'chemvista_ik2d', 'chemvista_evaluation_role', 'public_spectrum_id',
    'public_ik2d', 'same_ik2d', 'modified_cosine', 'matched_peaks',
])
if len(modified_cosine_pair_audit):
    best_pairs = modified_cosine_pair_audit.sort_values(
        ['chemvista_spectrum_id', 'modified_cosine', 'matched_peaks'], ascending=[True, False, False]
    ).drop_duplicates('chemvista_spectrum_id')
    best_score = best_pairs.set_index('chemvista_spectrum_id')['modified_cosine']
    best_matches = best_pairs.set_index('chemvista_spectrum_id')['matched_peaks']
    best_public = best_pairs.set_index('chemvista_spectrum_id')['public_spectrum_id']
else:
    best_score = pd.Series(dtype=float)
    best_matches = pd.Series(dtype=float)
    best_public = pd.Series(dtype='string')
chem_eval['max_modified_cosine_to_public_train'] = chem_eval['spectrum_id'].map(best_score).fillna(0.0)
chem_eval['matched_peaks_at_max_modified_cosine'] = chem_eval['spectrum_id'].map(best_matches).fillna(0).astype(int)
chem_eval['closest_public_train_spectrum_id'] = chem_eval['spectrum_id'].map(best_public).fillna('')
chem_eval['confirmed_near_public_train_duplicate'] = chem_eval['max_modified_cosine_to_public_train'].ge(0.90) & chem_eval['matched_peaks_at_max_modified_cosine'].ge(6)
is_novel_spectrum = chem_eval['chemvista_evaluation_role'].isin(['natural_novel', 'selected_novel'])
chem_eval['evaluation_eligible'] = ~chem_eval['exact_public_train_spectrum_collision'] & ~(is_novel_spectrum & chem_eval['confirmed_near_public_train_duplicate'])

cross_summary = chem_eval.groupby('chemvista_evaluation_role').agg(
    n_spectra=('spectrum_id', 'size'),
    n_compounds=('inchikey_2d', 'nunique'),
    exact_public_train_collisions=('exact_public_train_spectrum_collision', 'sum'),
    near_public_train_candidates=('near_public_train_candidate', 'sum'),
    confirmed_near_public_train_duplicates=('confirmed_near_public_train_duplicate', 'sum'),
    evaluation_eligible_spectra=('evaluation_eligible', 'sum'),
).reset_index()
display(cross_summary)

## 5. 覆盖率、质控、冻结输出

In [ ]:
coverage = split[['inchikey_2d', 'supervised_compound_role', 'chemvista_evaluation_role', 'cv_group', 'cv_fold']].copy()
for column in ['primary_train_eligible_20ppm', 'primary_train_eligible_50ppm', 'primary_train_eligible_100ppm', 'strict_train_eligible_20ppm']:
    counts = public.loc[public[column]].groupby('inchikey_2d').size()
    coverage[f'n_spectra__{column}'] = coverage['inchikey_2d'].map(counts).fillna(0).astype(int)
chem_counts = chem_eval.loc[chem_eval['evaluation_eligible']].groupby('inchikey_2d').size()
coverage['n_spectra__chemvista_evaluation'] = coverage['inchikey_2d'].map(chem_counts).fillna(0).astype(int)

task_columns = [c for c in split.columns if c.startswith('label__') and c in {
    'label__P_screen_not_ready_binary', 'label__B_LOGBCF_REGRESSION',
    'label__P_READY_BOD_D28_OECD301C_SOURCE_301C', 'label__T_EC10_MG_L_REGRESSION',
    'label__T_EC50_MG_L_REGRESSION', 'label__T_LC50_MG_L_REGRESSION', 'label__T_NOEC_MG_L_REGRESSION'}]
task_rows = []
for label_col in task_columns:
    labeled = split[label_col].notna()
    for subset, mask, spectrum_col in [
        ('development_20ppm', split['supervised_compound_role'].eq('development'), 'n_spectra__primary_train_eligible_20ppm'),
        ('strict_development_20ppm', split['supervised_compound_role'].eq('development') & ~split['chemvista_evaluation_role'].eq('transfer'), 'n_spectra__strict_train_eligible_20ppm'),
        ('chemvista_novel', split['chemvista_evaluation_role'].isin(['natural_novel', 'selected_novel']), 'n_spectra__chemvista_evaluation'),
        ('chemvista_transfer', split['chemvista_evaluation_role'].eq('transfer'), 'n_spectra__chemvista_evaluation'),
    ]:
        selected = coverage.index[labeled & mask]
        n_with_spectra = int(coverage.loc[selected, spectrum_col].gt(0).sum())
        task_rows.append({'task': label_col.removeprefix('label__'), 'subset': subset, 'n_labeled_compounds': len(selected), 'n_with_eligible_spectra': n_with_spectra})
task_coverage = pd.DataFrame(task_rows)

checks = []
def add_check(check_id, description, passed, details='', warn=False):
    checks.append({'check_id': check_id, 'description': description, 'status': 'PASS' if passed else ('WARN' if warn else 'FAIL'), 'details': details})

novel_keys = set(split.loc[split['chemvista_evaluation_role'].isin(['natural_novel', 'selected_novel']), 'inchikey_2d'])
transfer_keys = set(split.loc[split['chemvista_evaluation_role'].eq('transfer'), 'inchikey_2d'])
primary_train_keys = set(public.loc[public['primary_train_eligible_20ppm'], 'inchikey_2d'])
strict_train_keys = set(public.loc[public['strict_train_eligible_20ppm'], 'inchikey_2d'])
all_chemvista_eval_keys = novel_keys | transfer_keys
eligible_novel_hashes = set(chem_eval.loc[chem_eval['evaluation_eligible'] & chem_eval['chemvista_evaluation_role'].isin(['natural_novel', 'selected_novel']), 'spectrum_hash_common'])

add_check('SPEC001', '上一步冻结拆分质控全部通过', split_qc['status'].eq('PASS').all())
add_check('SPEC002', '公共流谱图总数与清洗摘要或legacy计数一致', all_offset_count == expected_count, f'{all_offset_count} / {expected_count}')
add_check('SPEC003', '随机访问记录IK2D无错位', key_mismatch_count == 0, str(key_mismatch_count))
add_check('SPEC004', '主训练20ppm不含Novel结构', primary_train_keys.isdisjoint(novel_keys), str(len(primary_train_keys & novel_keys)))
add_check('SPEC005', 'Strict训练20ppm不含任何ChemVista外测结构', strict_train_keys.isdisjoint(all_chemvista_eval_keys), str(len(strict_train_keys & all_chemvista_eval_keys)))
add_check('SPEC006', '主训练谱图均为去重保留且无跨身份精确碰撞', public.loc[public['primary_train_eligible_20ppm'], ['exact_dedup_keep']].all().all() and not public.loc[public['primary_train_eligible_20ppm'], 'cross_identity_exact_collision'].any())
add_check('SPEC007', '有效Novel谱图哈希与主训练无精确交集', eligible_novel_hashes.isdisjoint(primary_hashes), str(len(eligible_novel_hashes & primary_hashes)))
add_check('SPEC008', 'ChemVista任务结构均至少有一张可评价谱图', set(chem_eval.loc[chem_eval['evaluation_eligible'], 'inchikey_2d']) == all_chemvista_eval_keys, f"{chem_eval.loc[chem_eval['evaluation_eligible'], 'inchikey_2d'].nunique()} / {len(all_chemvista_eval_keys)}")
traceability_complete = bool(is_traceable_v2 and public['record_schema_version'].eq('ENVMOLBENCH_CLEANED_V2').all() and public['original_spectrum_id'].notna().all())
traceability_details = f"schema={public_schema_version}; original_id_nonnull={int(public['original_spectrum_id'].notna().sum())}/{len(public)}; instrument_nonnull={int(public['instrument'].notna().sum())}/{len(public)}"
if is_traceable_v2:
    add_check('SPEC009', '公共V2文件包含逐谱图来源、原始ID和仪器追溯字段', traceability_complete, traceability_details)
else:
    add_check('SPEC009', '公共legacy文件缺少完整原始谱图追溯字段', False, '历史导出未保存且不可从当前PKL无损恢复；来源仅按append rank审计恢复，不用于仪器分层推断', warn=True)
add_check('SPEC010', '主分析只使用identity ppm<=20的公共谱图', public.loc[public['primary_train_eligible_20ppm'], 'identity_ppm_error'].le(20).all())
eligible_novel = chem_eval['evaluation_eligible'] & chem_eval['chemvista_evaluation_role'].isin(['natural_novel', 'selected_novel'])
add_check('SPEC011', '有效Novel谱图无modified-cosine确认近重复', not chem_eval.loc[eligible_novel, 'confirmed_near_public_train_duplicate'].any())
add_check('SPEC012', '全部粗签名候选均完成modified-cosine确认', chem_eval.loc[chem_eval['near_public_train_candidate'], 'spectrum_id'].isin(set(modified_cosine_pair_audit['chemvista_spectrum_id'])).all())
qc = pd.DataFrame(checks)
assert not qc['status'].eq('FAIL').any(), qc[qc['status'].eq('FAIL')]

public = public.drop(columns=['library_priority'])
public.to_pickle(OUTPUT_DIR / 'public_relevant_spectrum_manifest.pkl')
public.to_csv(OUTPUT_DIR / 'public_relevant_spectrum_manifest.csv.gz', index=False, compression='gzip')
chem_eval.to_pickle(OUTPUT_DIR / 'chemvista_evaluation_spectrum_manifest.pkl')
chem_eval.to_csv(OUTPUT_DIR / 'chemvista_evaluation_spectrum_manifest.csv.gz', index=False, compression='gzip')
coverage.to_csv(OUTPUT_DIR / 'compound_spectrum_coverage.csv', index=False)
task_coverage.to_csv(OUTPUT_DIR / 'task_spectrum_coverage.csv', index=False)
dedup_summary.to_csv(OUTPUT_DIR / 'public_duplicate_audit_summary.csv', index=False)
cross_summary.to_csv(OUTPUT_DIR / 'public_chemvista_collision_summary.csv', index=False)
modified_cosine_pair_audit.to_csv(OUTPUT_DIR / 'public_chemvista_modified_cosine_pair_audit.csv.gz', index=False, compression='gzip')
qc.to_csv(OUTPUT_DIR / 'spectrum_audit_qc_checks.csv', index=False)

input_hashes = {}
hash_input_names = ['split_manifest_csv', 'split_freeze_json', 'split_qc_csv', 'public_index', 'public_cleaning_notebook', 'public_cleaning_module', 'chemvista_index', 'chemvista_peaks', 'chemvista_summary']
if is_traceable_v2:
    hash_input_names.append('public_summary')
for name in hash_input_names:
    input_hashes[name] = {'path': str(PATHS[name].relative_to(ROOT)), 'sha256': sha256_file(PATHS[name]), 'size_bytes': PATHS[name].stat().st_size}
public_stream_stat = PATHS['public_stream'].stat()
public_stream_sha256 = sha256_file(PATHS['public_stream'])
audit_manifest = {
    'audit_version': 'PBT_MSMS_SPECTRUM_AUDIT_V3',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'split_version': split_freeze['split_version'],
    'input_hashes': input_hashes,
    'public_stream': {'path': str(PATHS['public_stream'].relative_to(ROOT)), 'size_bytes': public_stream_stat.st_size, 'sha256': public_stream_sha256, 'header': stream_header},
    'library_counts': dict(LIBRARY_COUNTS),
    'source_library_provenance': ({'method': 'stored per spectrum by public cleaner V2', 'validation': 'counts agree with cleaner summary and stream/index count', 'limitation': None} if is_traceable_v2 else {'method': 'legacy append-rank boundaries', 'validation': 'total accepted counts exactly match stream/index count', 'limitation': 'historical SELECT had no explicit ORDER BY; source_library is descriptive only and not used for inferential stratification'}),
    'counts': {
        'public_relevant_spectra': len(public),
        'public_relevant_compounds': int(public['inchikey_2d'].nunique()),
        'primary_train_20ppm_spectra': int(public['primary_train_eligible_20ppm'].sum()),
        'primary_train_20ppm_compounds': int(public.loc[public['primary_train_eligible_20ppm'], 'inchikey_2d'].nunique()),
        'strict_train_20ppm_spectra': int(public['strict_train_eligible_20ppm'].sum()),
        'strict_train_20ppm_compounds': int(public.loc[public['strict_train_eligible_20ppm'], 'inchikey_2d'].nunique()),
        'chemvista_evaluation_spectra': len(chem_eval),
        'chemvista_evaluation_compounds': int(chem_eval['inchikey_2d'].nunique()),
        'coarse_near_duplicate_candidates': int(chem_eval['near_public_train_candidate'].sum()),
        'confirmed_modified_cosine_near_duplicates': int(chem_eval['confirmed_near_public_train_duplicate'].sum()),
        'confirmed_novel_near_duplicates_excluded': int((is_novel_spectrum & chem_eval['confirmed_near_public_train_duplicate']).sum()),
    },
    'hash_definition': {'mz_decimals': 4, 'intensity_decimals': 5, 'precursor_decimals': 4, 'max_fragments': 119, 'isotope_filter': True},
    'near_duplicate_definition': {'candidate': 'same coarse precursor/ion/top20 peak signature', 'confirmation': 'sqrt-intensity modified cosine >=0.90 with >=6 matched peaks at 0.02 Da tolerance', 'novel_policy': 'confirmed near duplicates excluded from evaluation', 'transfer_policy': 'reported but retained unless exact hash collision'},
    'legacy_limitations': ([] if is_traceable_v2 else ['original public spectrum ID unavailable', 'public instrument metadata unavailable', 'raw public peaks unavailable; cleaned tensor peaks used']),
    'software': {'python': sys.version, 'executable': sys.executable, 'pandas': pd.__version__, 'numpy': np.__version__, 'rdkit': rdBase.rdkitVersion, 'platform': platform.platform()},
    'qc_status': 'PASS_WITH_WARNINGS' if qc['status'].eq('WARN').any() else 'PASS',
}
with (OUTPUT_DIR / 'spectrum_audit_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(audit_manifest, handle, ensure_ascii=False, indent=2)
    handle.write('\n')

display(qc)
display(task_coverage)
print(json.dumps(audit_manifest['counts'], ensure_ascii=False, indent=2))
print('审计状态:', audit_manifest['qc_status'])